# Compress VGG-16-BN (ImageNet, 528 MB) — Kaggle notebook

`torchvision VGG-16-BN (pretrained)  →  gradual prune + fine-tune  →  INT8 quantize  →  ONNX  →  verify against the original`

The step up from the CIFAR notebooks: **138 M parameters, 528 MB, 224×224 inputs**. The same pipeline, no model-specific code.

**Before Run All** (right panel): *Accelerator → GPU T4*, *Internet → On* (downloads the pretrained weights),
and *Add Input* an ImageNet dataset laid out as `train/<class>/*` and `val/<class>/*` with **all 1000 class folders**
(the cell below searches `/kaggle/input` and stops with a clear message if it finds none).

Accuracy is measured on a fixed 2000-image slice of the validation set (the full 50,000 on CPU would take hours at every stage),
so differences under about 1 pp are noise.

Everything lands in `/kaggle/working/` — download `vgg16_compressed.zip` from the *Output* tab.

In [ ]:
# ======================= CONFIG =======================
ARCH        = "tv:vgg16_bn"   # any torchvision ImageNet model: tv:resnet152, tv:vgg19_bn, ...
PRUNE_RATIO = 0.5    # fraction of channels removed (0.5 -> ~75% fewer parameters)
STEPS       = 3      # gradual pruning steps
FT_EPOCHS   = 1      # fine-tune epochs after every step (each epoch is a pass over the train split)
LOSS        = "dkd"  # "kd" | "dkd"
BATCH       = 32
CALIB       = 256    # calibration images for INT8 (train split)
TEST_IMAGES = 2000   # validation images used to score every stage
# ======================================================

In [ ]:
import os, subprocess, sys, json, shutil, pathlib
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists("/kaggle/working/ck"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/ck"], check=True)
os.chdir("/kaggle/working/ck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> fine-tuning will run on CPU and take very long; enable a GPU, then Restart & Run All")

def find_imagenet_root(base="/kaggle/input"):
    """First directory that has train/ and val/ with 1000 class folders each."""
    for dirpath, dirnames, _ in os.walk(base):
        if {"train", "val"} <= set(dirnames):
            counts = [sum(os.path.isdir(os.path.join(dirpath, s, d)) for d in os.listdir(os.path.join(dirpath, s)))
                      for s in ("train", "val")]
            if counts == [1000, 1000]:
                return dirpath
        if dirpath.count(os.sep) - base.count(os.sep) >= 4:
            dirnames[:] = []
    return None

DATA_ROOT = find_imagenet_root()
assert DATA_ROOT, ("No ImageNet-layout dataset under /kaggle/input. Add one with train/<class>/ and val/<class>/ "
                   "(all 1000 classes) via Add Input, then re-run.")
print("ImageNet root:", DATA_ROOT)

def run(*args):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    return subprocess.run(args).returncode

## 1 · Save the pretrained original (the reference to verify against)

In [ ]:
from compresskit.zoo.torchvision_models import build_torchvision
from compresskit.zoo.train import save_model
original = build_torchvision(ARCH)
os.makedirs("checkpoints", exist_ok=True)
CKPT = "checkpoints/vgg16_bn_imagenet.pt"
save_model(original, CKPT, {"arch": ARCH, "dataset": "imagenet", "input_shape": [1, 3, 224, 224]})
print(f"{sum(p.numel() for p in original.parameters()):,} parameters -> {CKPT}")

## 2 · Compress: gradual prune + fine-tune → INT8 → evaluate the .onnx

In [ ]:
OUT = "/kaggle/working"
code_ = run(sys.executable, "-m", "compresskit.cli", "pipeline",
            "--model", CKPT, "--dataset", "imagenet", "--data-root", DATA_ROOT,
            "--test-samples", TEST_IMAGES, "--calib", CALIB, "--batch-size", BATCH,
            "--prune", PRUNE_RATIO, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS, "--loss", LOSS,
            "--precision", "int8", "--allow-slow-int8",
            "--workdir", f"{OUT}/work", "--report", f"{OUT}/report.html", "--json", f"{OUT}/results.json")
assert code_ == 0, "pipeline failed - read the log above"

## 3 · Does the compressed model actually work? (original vs ONNX on real validation images)

In [ ]:
r = json.load(open(f"{OUT}/results.json"))
shutil.copy(r["stages"][-1]["artifact"], f"{OUT}/model.onnx")
run(sys.executable, "experiments/verify_onnx.py", "--model", CKPT, "--onnx", f"{OUT}/model.onnx",
    "--dataset", "imagenet", "--data-root", DATA_ROOT, "--images", 1000)

## 4 · Summary and download bundle

In [ ]:
b, f, t = r["baseline"], r["stages"][-1], r["totals"]
print(f"parameters: {b['parameters']:,} -> {f['parameters']:,}")
print(f"accuracy  : {b['accuracy']*100:.2f}% -> {f['accuracy']*100:.2f}%   (drop {t['accuracy_drop_pp']:.2f} pp, on {TEST_IMAGES} val images)")
print(f"size      : {b['size_mb']:.1f} MB -> {f['size_mb']:.1f} MB   ({t['size_reduction']:.1f}x smaller)")
print(f"precision : {f['precision']}")
print("NOTE: latency in this run is Kaggle-CPU latency, not your device. Measure on the target: compresskit bench model.onnx --input-shape 1 3 224 224")

import zipfile
with zipfile.ZipFile(f"{OUT}/vgg16_compressed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.onnx", "report.html", "results.json"):
        z.write(f"{OUT}/{name}", name)
print("bundle:", f"{OUT}/vgg16_compressed.zip", f"({os.path.getsize(f'{OUT}/vgg16_compressed.zip')/1e6:.1f} MB)")

## 5 · On your Mac (real speed check, once)

```bash
unzip vgg16_compressed.zip
compresskit bench model.onnx --input-shape 1 3 224 224
```